# Columnar Transformer

In [ ]:
# Jab hum data par preprocessing lagate hain, to har column ke liye alag rules ya instructions define karte hain. In sab instructions ko scattered rakhne ki jagah hum ek container ke andar organize kar dete hain.

# `ColumnTransformer` ko ek template ya blueprint ki tarah samajh sakte hain. Jaise OOP me hum ek class banate hain jisme kuch rules aur methods define hote hain, aur phir us class ko baar-baar use karte hain. Waise hi `ColumnTransformer` me hum define karte hain ki:

# * Numeric columns par kya processing hogi (scaling, imputation, etc.)
# * Categorical columns par kya processing hogi (encoding, imputation, etc.)
# * Aur baaki columns ke saath kya karna hai

# Is tarah saari preprocessing instructions ek hi jagah neatly organized rehti hain. Jab bhi data aata hai, `ColumnTransformer` us blueprint ko follow karke har column par uski corresponding transformation apply kar deta hai.

# Yaani, `ColumnTransformer` ek centralized preprocessing template hai jo different column groups ke liye different transformations ko manage aur execute karta hai.


In [3]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.impute import SimpleImputer # it is used to fill the missing values
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [9]:
df = pd.read_csv(r'C:\Users\PIYUSH\OneDrive\Desktop\Data-Science\Datasets\covid_toy - covid_toy.csv')

In [10]:
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [11]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [12]:
x = df.drop(columns = 'has_covid')
y = df['has_covid']

In [15]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size= .2,random_state=42)

In [16]:
x_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


# Manully typed input


In [19]:
# adding simple imputer to the fever
si = SimpleImputer(strategy = 'mean')

In [22]:
x_train_fever = si.fit_transform(x_train[['fever']])
x_test_fever = si.fit_transform(x_test[['fever']])

In [24]:
x_train_fever.shape

(80, 1)

In [28]:
# ordinal Encoding
oe = OrdinalEncoder(categories = [['Mild','Strong']])
x_train_cough = oe.fit_transform(x_train[['cough']])

# also test the data
x_test_cough = oe.fit_transform(x_test[['cough']])
x_train_cough.shape

(80, 1)

# One Hot Encoding

 gender , city
 

In [29]:
ohe = OneHotEncoder(drop = 'first',sparse_output = False)
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])

# also test the data
x_test_gender_city = ohe.fit_transform(x_test[['gender','city']])
x_train_gender_city.shape

(80, 4)

In [31]:
# Extracting the age

x_train_age = x_train.drop(columns = ['gender','cough','city','fever']).values

x_test_age = x_test.drop(columns = ['gender','fever','cough','city']).values

In [32]:
x_train_age.shape

(80, 1)

In [33]:
x_train_transformed = np.concatenate((x_train_age,x_train_fever,
x_train_gender_city,x_train_cough),axis = 1)

In [34]:
x_train_transformed.shape

(80, 7)

In [35]:
# by the help of columnar transformer

In [ ]:
from sklearn.compose import ColumnTransformer # how to import columns transformer
transformer = ColumnTransformer(transformers=[ 
# in the fever columsn with the help of simple imputer we fill missing values by mean,median,mode
('tnf1',SimpleImputer(),['fever']),
# by this proecess we encode our data
('tmf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
# remainder = passthrough ==> 
# means rest all the columns remain same
('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])]
,remainder='passthrough'

)

In [38]:
transformer.fit_transform(x_train).shape

(80, 7)

In [40]:
transformer.transform(x_test).shape

(20, 7)

In [42]:
# in hamara jo data set hai jiske upar apan proecessing kr rhe hai vo proecessing bina kisi containarized or
# containzer se same tarike se kr sakte hai
# columnar transformer ek template bna deta hai jha step of processes ek structure tarike se rke hue ho
# we pass three things into the columsn transformer 
# 1. constant, 2.approch ya logic name # vo approch ya logic konse columns p impleemnt ho rhi hai